In [8]:
# ============================================================
# STABLE GENE-SPECIFIC COX FEATURES
# ============================================================

import pandas as pd
import numpy as np
from lifelines import CoxPHFitter


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

MIN_GENE_PATIENTS = 20
P_VALUE_THRESHOLD = 0.05


# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------

# target:
# ID | OS_YEARS | OS_STATUS

# molecular:
# ID | CHR | START | END | REF | ALT | GENE | ...

patient_data = target[
    ["ID", "OS_YEARS", "OS_STATUS"]
].copy()


# Make sure survival variables are numeric
patient_data["OS_YEARS"] = pd.to_numeric(
    patient_data["OS_YEARS"],
    errors="coerce"
)

patient_data["OS_STATUS"] = pd.to_numeric(
    patient_data["OS_STATUS"],
    errors="coerce"
)


# Remove invalid survival records
patient_data = patient_data.dropna(
    subset=["ID", "OS_YEARS", "OS_STATUS"]
)


# ============================================================
# GENE-SPECIFIC SURVIVAL EFFECTS
# ============================================================

gene_effects = []


# Number of patients in dataset
n_patients = patient_data["ID"].nunique()

print("Number of patients:", n_patients)


# ------------------------------------------------------------
# IMPORTANT:
# Count UNIQUE patients per gene.
#
# A patient can have the same gene more than once because
# they may have multiple mutations in that gene.
# ------------------------------------------------------------

gene_patient_counts = (
    molecular
    .dropna(subset=["GENE", "ID"])
    .drop_duplicates(["ID", "GENE"])
    .groupby("GENE")["ID"]
    .nunique()
    .sort_values(ascending=False)
)


print("\nGene mutation frequencies:")
print(gene_patient_counts.head(20))


# ------------------------------------------------------------
# Loop through genes
# ------------------------------------------------------------

for gene, n_mutated in gene_patient_counts.items():

    # --------------------------------------------------------
    # Skip very rare genes
    # --------------------------------------------------------

    if n_mutated < MIN_GENE_PATIENTS:
        continue


    # --------------------------------------------------------
    # Patients carrying this gene
    # --------------------------------------------------------

    mutated_patients = set(
        molecular.loc[
            molecular["GENE"] == gene,
            "ID"
        ].dropna()
    )


    # --------------------------------------------------------
    # Create patient-level binary mutation variable
    # --------------------------------------------------------

    temp = patient_data.copy()

    temp["gene_mutated"] = (
        temp["ID"].isin(mutated_patients)
    ).astype(int)


    # --------------------------------------------------------
    # Check that both groups exist
    # --------------------------------------------------------

    if temp["gene_mutated"].nunique() < 2:
        continue


    # --------------------------------------------------------
    # Number of deaths among mutated patients
    # --------------------------------------------------------

    mutated = temp[
        temp["gene_mutated"] == 1
    ]

    non_mutated = temp[
        temp["gene_mutated"] == 0
    ]


    n_deaths_mutated = (
        mutated["OS_STATUS"].sum()
    )

    n_deaths_non_mutated = (
        non_mutated["OS_STATUS"].sum()
    )


    # --------------------------------------------------------
    # Skip genes with no events in either group
    #
    # Cox models need survival events to estimate an effect.
    # --------------------------------------------------------

    if n_deaths_mutated == 0:
        continue

    if n_deaths_non_mutated == 0:
        continue


    # --------------------------------------------------------
    # Prepare Cox data
    # --------------------------------------------------------

    cox_data = temp[
        [
            "OS_YEARS",
            "OS_STATUS",
            "gene_mutated"
        ]
    ].dropna()


    if len(cox_data) < 50:
        continue


    # --------------------------------------------------------
    # Penalized Cox model
    #
    # The penalizer stabilizes estimates for relatively rare
    # genes and reduces extreme coefficients.
    # --------------------------------------------------------

    try:

        cph = CoxPHFitter(
            penalizer=0.1
        )


        cph.fit(
            cox_data,
            duration_col="OS_YEARS",
            event_col="OS_STATUS"
        )


        # ----------------------------------------------------
        # Extract coefficient
        # ----------------------------------------------------

        coefficient = float(
            cph.params_["gene_mutated"]
        )


        # ----------------------------------------------------
        # Prevent numerical overflow
        #
        # We primarily use log(HR), so this is also the most
        # stable representation for ML.
        # ----------------------------------------------------

        log_hr = coefficient


        # Calculate HR safely
        if log_hr > 20:

            hr = np.exp(20)

        elif log_hr < -20:

            hr = np.exp(-20)

        else:

            hr = np.exp(log_hr)


        # ----------------------------------------------------
        # P-value
        # ----------------------------------------------------

        pvalue = float(
            cph.summary.loc[
                "gene_mutated",
                "p"
            ]
        )


        # ----------------------------------------------------
        # Confidence interval
        #
        # lifelines stores these on the log-HR scale.
        # ----------------------------------------------------

        ci_lower_log = float(
            cph.confidence_intervals_.loc[
                "gene_mutated",
                "95% lower-bound"
            ]
        )

        ci_upper_log = float(
            cph.confidence_intervals_.loc[
                "gene_mutated",
                "95% upper-bound"
            ]
        )


        # Safe exponential
        ci_lower = np.exp(
            np.clip(ci_lower_log, -20, 20)
        )

        ci_upper = np.exp(
            np.clip(ci_upper_log, -20, 20)
        )


        # ----------------------------------------------------
        # Direction
        #
        # HR > 1 = higher hazard
        # HR < 1 = lower hazard
        #
        # We only assign a direction when statistically
        # significant.
        # ----------------------------------------------------

        if pvalue < P_VALUE_THRESHOLD:

            if log_hr > 0:

                direction = -1

            elif log_hr < 0:

                direction = 1

            else:

                direction = 0

        else:

            direction = 0


        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        gene_effects.append({

            "GENE": gene,

            "n_gene_patients": n_mutated,

            "n_gene_deaths": int(n_deaths_mutated),

            "gene_log_HR": log_hr,

            "gene_HR": hr,

            "gene_pvalue": pvalue,

            "gene_CI_lower": ci_lower,

            "gene_CI_upper": ci_upper,

            "gene_effect_direction": direction

        })


    except Exception as e:

        print(
            f"Skipped {gene}: {str(e)}"
        )


# ============================================================
# RESULTS
# ============================================================

gene_effects = pd.DataFrame(
    gene_effects
)


print("\nNumber of genes with stable Cox estimates:")
print(len(gene_effects))


# Sort by p-value
if len(gene_effects) > 0:

    print(
        gene_effects
        .sort_values("gene_pvalue")
        .head(20)
    )

# ============================================================
# MERGE GENE EFFECTS INTO MOLECULAR DATA
# ============================================================

# Remove gene-effect columns if this cell was already run before.
# This prevents columns such as n_gene_patients_x / n_gene_patients_y.
gene_effect_columns = [
    "n_gene_patients",
    "n_gene_deaths",
    "gene_log_HR",
    "gene_HR",
    "gene_pvalue",
    "gene_CI_lower",
    "gene_CI_upper",
    "gene_effect_direction"
]

molecular = molecular.drop(
    columns=[
        col for col in gene_effect_columns
        if col in molecular.columns
    ],
    errors="ignore"
)


# ------------------------------------------------------------
# Merge the gene-level Cox results
# ------------------------------------------------------------

molecular = molecular.merge(
    gene_effects,
    on="GENE",
    how="left"
)


# ============================================================
# FILL MISSING VALUES
# ============================================================

# Genes that were too rare or could not be fitted
# receive neutral/default values.

molecular["n_gene_patients"] = (
    molecular["n_gene_patients"]
    .fillna(0)
    .astype(int)
)

molecular["n_gene_deaths"] = (
    molecular["n_gene_deaths"]
    .fillna(0)
    .astype(int)
)

# log(HR) = 0 means no estimated effect
molecular["gene_log_HR"] = (
    molecular["gene_log_HR"]
    .fillna(0.0)
)

# HR = 1 means no estimated effect
molecular["gene_HR"] = (
    molecular["gene_HR"]
    .fillna(1.0)
)

# p = 1 means no statistical evidence
molecular["gene_pvalue"] = (
    molecular["gene_pvalue"]
    .fillna(1.0)
)

# Direction:
# +1 = lower hazard / positive survival association
# -1 = higher hazard / negative survival association
#  0 = neutral / not statistically significant
molecular["gene_effect_direction"] = (
    molecular["gene_effect_direction"]
    .fillna(0)
    .astype(int)
)


# ============================================================
# CHECK
# ============================================================

print("\nMolecular shape:")
print(molecular.shape)

print("\nGene-effect columns:")
print(
    molecular[
        [
            "GENE",
            "n_gene_patients",
            "n_gene_deaths",
            "gene_log_HR",
            "gene_HR",
            "gene_pvalue",
            "gene_effect_direction"
        ]
    ].head(20)
)

Number of patients: 3173

Gene mutation frequencies:
GENE
TET2      1033
ASXL1      893
SF3B1      739
SRSF2      571
DNMT3A     534
RUNX1      462
TP53       379
STAG2      301
U2AF1      285
EZH2       216
BCOR       183
CBL        180
ZRSR2      177
NRAS       174
IDH2       166
CUX1       127
NF1        124
KRAS       123
SETBP1     121
DDX41      118
Name: ID, dtype: int64

Number of genes with stable Cox estimates:
64
      GENE  n_gene_patients  n_gene_deaths  gene_log_HR   gene_HR  \
6     TP53              379            271     0.874931  2.398710   
5    RUNX1              462            309     0.655082  1.925300   
1    ASXL1              893            534     0.434325  1.543921   
7    STAG2              301            198     0.581195  1.788175   
13    NRAS              174            119     0.648800  1.913243   
9     EZH2              216            141     0.563592  1.756973   
2    SF3B1              739            311    -0.370680  0.690264   
22     MLL          